# GRL Manuscript Figures (NB06)
**Figures 1–4 for:** *Circumpolar Satellite Evidence for Topographically-Modulated Multi-Decadal Evolution of Southern Ocean Standing Meanders*

Run each cell sequentially. Adjust parameters in-line as needed.


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import Normalize
import warnings

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print("WARNING: Cartopy not available. Install with: pip install cartopy")

# Nature-style defaults
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9, "axes.linewidth": 0.8, "axes.labelsize": 10,
    "axes.titlesize": 11, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "legend.framealpha": 0.9, "figure.dpi": 150, "mathtext.default": "regular",
})
print("Imports and rcParams loaded.")


In [ ]:
# ══════════════ PATHS ══════════════
from pathlib import Path

BASE_DIR = Path("/g/data/gv90/xl1657/cmems_adt/grl_meander_products")
GMRT_DIR = Path("/g/data/gv90/xl1657/gmrt")
FIG_DIR  = BASE_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════ COLOUR-BLIND SAFE PALETTE (Wong 2011) ══════════════
CB = {"CP": "#0072B2", "PAR": "#E69F00", "SEIR": "#CC79A7", "SWIR": "#009E73"}

SITES = {
    "CP":   {"name": "Campbell Plateau",        "short": "CP",   "color": CB["CP"],
             "inner_lon": (150, -150), "inner_lat": (-57, -46), "wraps": True},
    "PAR":  {"name": "Pacific-Antarctic Ridge",  "short": "PAR",  "color": CB["PAR"],
             "inner_lon": (-150, -80), "inner_lat": (-60, -48), "wraps": False},
    "SEIR": {"name": "Southeast Indian Ridge",   "short": "SEIR", "color": CB["SEIR"],
             "inner_lon": (130, 152),  "inner_lat": (-56, -44), "wraps": False},
    "SWIR": {"name": "Southwest Indian Ridge",   "short": "SWIR", "color": CB["SWIR"],
             "inner_lon": (15, 45),    "inner_lat": (-58, -44), "wraps": False},
}

GMRT_FILES = {
    "CP_east": GMRT_DIR / "GMRTv4_4_1_20260314topo_CP_East.grd",
    "CP_west": GMRT_DIR / "GMRTv4_4_1_20260314topo_CP_West.grd",
    "PAR":     GMRT_DIR / "GMRTv4_4_1_20260314topo_PAR.grd",
    "SEIR":    GMRT_DIR / "GMRTv4_4_1_20260314topo_SEIR.grd",
    "SWIR":    GMRT_DIR / "GMRTv4_4_1_20260314topo_SWIR.grd",
}

DECADES = [
    ("1993\u20132002", "1993-01", "2002-12", "#4575b4", 1.0, "-"),
    ("2003\u20132014", "2003-01", "2014-12", "#7570b3", 1.6, "--"),
    ("2015\u20132025", "2015-01", "2025-12", "#d95f02", 2.2, "-"),
]

DEG_TO_KM = 111.32

def sig_label(p_val):
    return "*" if p_val < 0.05 else ""

print("Configuration loaded.")


In [ ]:
# ══════════════ DATA LOADERS ══════════════

def load_gmrt(fp, coarsen=4):
    """Load GMT-format .grd bathymetry file."""
    ds = xr.open_dataset(fp)
    nx, ny = int(ds["dimension"].values[0]), int(ds["dimension"].values[1])
    x0, x1 = ds["x_range"].values
    y0, y1 = ds["y_range"].values
    lon = np.linspace(x0, x1, nx)
    lat = np.linspace(y0, y1, ny)
    z = ds["z"].values.reshape(ny, nx)
    if lat[0] > lat[-1]:
        lat = lat[::-1]
        z = z[::-1, :]
    ds.close()
    if coarsen > 1:
        lon, lat, z = lon[::coarsen], lat[::coarsen], z[::coarsen, ::coarsen]
    return lon, lat, z


def load_all_data():
    print("Loading data...")
    data = {}
    for key in SITES:
        fp = BASE_DIR / f"monthly_metrics_{key}.csv"
        if fp.exists():
            data[key] = pd.read_csv(fp, index_col=0, parse_dates=True)
            print(f"  {key}: {len(data[key])} months")
    for name, tag in [("trend_results_mannkendall.csv", "trends"),
                      ("era5_wind_trends.csv", "wind_trends"),
                      ("argo_temperature_summary.csv", "argo")]:
        fp = BASE_DIR / name
        if fp.exists():
            data[tag] = pd.read_csv(fp)
    for key in SITES:
        fp = BASE_DIR / f"era5_wind_{key}.csv"
        if fp.exists():
            data[f"wind_{key}"] = pd.read_csv(fp, index_col=0, parse_dates=True)
    for key in SITES:
        fp = BASE_DIR / f"meander_detection_{key}_rel20_x4m.nc"
        if fp.exists():
            data[f"nc_{key}"] = xr.open_dataset(fp)
    return data

data = load_all_data()


## Figure 1: Circumpolar Map

**Adapted from MATLAB scripts:** `f_derive_meander_location_daily_origin.m` and `f_derive_meander_geos_speed_trend_original.m`

Key elements from MATLAB:
- `pcolor` of meander frequency field as background (Fig 5/6 in MATLAB script)
- Bathymetry contours at 1000 m intervals: `contour(mx, my, mz2, [-3000:1000:0])`
- Meander centre position (black), N/S edges (red)
- Coastline as filled black polygons
- Long-term mean position (thick grey), decadal means

**Python adaptation:** South polar stereographic projection, `pcolormesh` for bathymetry with `ocean` colormap, decadal mean meander positions at each site, bathymetry contours overlaid.


In [ ]:
# ══════════════ FIGURE 1: CIRCUMPOLAR MAP ══════════════
# Adapted from MATLAB: pcolor bathymetry, contour lines, meander positions,
# coastline polygons, frequency colorbar

if not HAS_CARTOPY:
    raise ImportError("Cartopy required for Figure 1. Install: pip install cartopy")

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(111, projection=ccrs.SouthPolarStereo())
ax.set_extent([-180, 180, -68, -35], ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="0.82", edgecolor="0.3", linewidth=0.5)
ax.coastlines(resolution="50m", linewidth=0.4, color="0.3")
tr = ccrs.PlateCarree()

# Gridlines with lon/lat labels
gl = ax.gridlines(crs=tr, draw_labels=True, linewidth=0.15,
                  color="gray", alpha=0.4, linestyle="--")
gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 30))
gl.ylocator = mticker.FixedLocator(np.arange(-65, -30, 5))
gl.xlabel_style = {"size": 7, "color": "0.4"}
gl.ylabel_style = {"size": 7, "color": "0.4"}
gl.top_labels = False

# ── Bathymetry: pcolormesh + contour lines (MATLAB: contour at -3000:1000:0) ──
pcm = None
for site_key in SITES:
    parts = ["CP_east", "CP_west"] if site_key == "CP" else [site_key]
    for part in parts:
        fp = GMRT_FILES.get(part)
        if fp and fp.exists():
            try:
                lon_b, lat_b, z_b = load_gmrt(fp, coarsen=6)
                LON, LAT = np.meshgrid(lon_b, lat_b)
                # pcolormesh — matches MATLAB pcolor
                pcm = ax.pcolormesh(LON, LAT, z_b, transform=tr,
                                   cmap="ocean", vmin=-6000, vmax=0,
                                   shading="auto", alpha=0.65, rasterized=True)
                # Contour lines at 1000 m intervals — matches MATLAB:
                # contour(mx, my, mz2, [-3000:1000:0], 'color', 'Black', 'LineWidth', 0.5)
                ax.contour(lon_b, lat_b, z_b, levels=np.arange(-5000, 0, 1000),
                          colors="0.3", linewidths=0.3, alpha=0.5, transform=tr)
            except Exception as e:
                print(f"  Warning: {fp.name}: {e}")

# Colorbar
if pcm is not None:
    cbar = fig.colorbar(pcm, ax=ax, orientation="horizontal",
                       fraction=0.035, pad=0.06, shrink=0.55, aspect=30)
    cbar.set_label("Depth (m)", fontsize=9)
    cbar.ax.tick_params(labelsize=7)

# ── Site domain boxes (solid rectangles) + full name labels ──
# Label positions tuned to avoid overlap
label_pos = {
    "CP":   (180,  -42, "center"),
    "PAR":  (-115, -43, "center"),
    "SEIR": (141,  -40, "center"),
    "SWIR": (30,   -39, "center"),
}

for site_key, site in SITES.items():
    lo0, lo1 = site["inner_lon"]
    la0, la1 = site["inner_lat"]
    col = site["color"]

    if not site.get("wraps", False):
        ax.plot([lo0, lo1, lo1, lo0, lo0],
                [la0, la0, la1, la1, la0],
                color=col, lw=2.0, ls="-", transform=tr)
    else:
        for xs, ys in [([lo0, 180], [la0, la0]), ([-180, lo1], [la0, la0]),
                       ([lo0, 180], [la1, la1]), ([-180, lo1], [la1, la1]),
                       ([lo0, lo0], [la0, la1]), ([lo1, lo1], [la0, la1])]:
            ax.plot(xs, ys, color=col, lw=2.0, ls="-", transform=tr)

    lp = label_pos[site_key]
    ax.text(lp[0], lp[1], site["name"], fontsize=7.5, fontweight="bold",
           color=col, ha=lp[2], va="center", transform=tr,
           bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=col, alpha=0.92, lw=0.8))

# ── Decadal meander positions (only at 4 sites) ──
# Adapted from MATLAB: plot(meand_loc_lon, meand_loc_lat_rollmean, ...)
for site_key in SITES:
    nc_key = f"nc_{site_key}"
    if nc_key not in data:
        continue
    ds = data[nc_key]
    lon_inner = ds["longitude"].values
    months = pd.DatetimeIndex(ds["month"].values)
    for lbl, ds0, ds1, dcol, dlw, dls in DECADES:
        mask = (months >= pd.Timestamp(ds0)) & (months <= pd.Timestamp(ds1))
        if mask.sum() == 0:
            continue
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            ml = np.nanmean(ds["center_lat"].values[mask, :], axis=0)
        v = np.isfinite(ml)
        if v.sum() > 0:
            ax.plot(lon_inner[v], ml[v], color=dcol, lw=dlw, ls=dls,
                   alpha=0.85, transform=tr, solid_capstyle="round")

# Legend
legend_els = [plt.Line2D([0], [0], color=c, lw=w, ls=ls, label=l)
              for l, _, _, c, w, ls in DECADES]
ax.legend(handles=legend_els, loc="lower left", fontsize=7.5,
         framealpha=0.95, title="Decadal mean position", title_fontsize=7.5)

plt.savefig(FIG_DIR / "fig1_circumpolar_map.pdf", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "fig1_circumpolar_map.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved to {FIG_DIR}")


## Figure 2: Trend Time Series
4-panel figure: position, width, speed, EKE with Sen's slope dashed lines and annotated statistics.


In [ ]:
# ══════════════ FIGURE 2: TREND TIME SERIES ══════════════
metrics = [
    ("center_lat", "Degree of latitude", "\u00b0/dec", True),
    ("width_km",   "Width anomaly (km)", "km/dec", False),
    ("speed_m_s",  r"Speed anomaly (m s$^{-1}$)", r"m s$^{-1}$/dec", False),
    ("eke_m2_s2",  r"EKE anomaly (m$^{2}$ s$^{-2}$)", r"m$^{2}$ s$^{-2}$/dec", False),
]
trends_df = data.get("trends")
all_years = list(range(1993, 2026))
fig, axes = plt.subplots(4, 1, figsize=(8, 12), sharex=True)

for j, (metric, ylabel, unit_str, add_km) in enumerate(metrics):
    ax = axes[j]
    anns = []
    for site_key, site in SITES.items():
        if site_key not in data or metric not in data[site_key].columns:
            continue
        y = data[site_key][metric].dropna()
        if len(y) == 0:
            continue
        y_anom = y - y.groupby(y.index.month).transform("mean")
        y_ann = y_anom.resample("YS").mean()
        ax.plot(y_ann.index, y_ann.values, color=site["color"], lw=1.3,
               label=site["name"], marker="o", ms=2.5, zorder=3)
        if trends_df is not None:
            row = trends_df[(trends_df["site"] == site_key) & (trends_df["metric"] == metric)]
            if len(row) > 0:
                slope = row.iloc[0]["slope_per_decade"]
                p_val = row.iloc[0]["p_value"]
                mid = len(y_ann) // 2
                x_yr = np.arange(len(y_ann))
                ax.plot(y_ann.index, (slope / 10) * (x_yr - mid),
                       color=site["color"], lw=1.0, ls="--", alpha=0.5, zorder=2)
                sig = sig_label(p_val)
                sign = "+" if slope > 0 else ""
                if add_km:
                    km_val = slope * DEG_TO_KM
                    km_sign = "+" if km_val > 0 else ""
                    txt = f"{site['name']}: {sign}{slope:.2f} {unit_str} ({km_sign}{km_val:.1f} km/dec)"
                else:
                    txt = f"{site['name']}: {sign}{slope:.3g} {unit_str}"
                if sig:
                    txt += f" {sig}"
                anns.append((txt, site["color"]))
    ax.set_ylabel(ylabel, fontsize=9)
    ax.tick_params(labelsize=8)
    ax.text(0.015, 0.93, f"({chr(97+j)})", transform=ax.transAxes,
           fontsize=12, fontweight="bold", va="top")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    # Store annotations
    ax._stat_anns = anns

axes[-1].set_xlabel("Year", fontsize=10)
axes[-1].set_xticks([pd.Timestamp(f"{y}-01-01") for y in all_years])
axes[-1].set_xticklabels([str(y) for y in all_years], rotation=45, ha="right", fontsize=6)

fig.tight_layout(h_pad=1.8, rect=[0, 0.10, 1, 1])

# Place statistics below each panel
for ax in axes:
    if hasattr(ax, "_stat_anns"):
        for idx, (txt, col) in enumerate(ax._stat_anns):
            ax.text(1.0, -0.18 - idx * 0.09, txt, transform=ax.transAxes,
                   fontsize=6, va="top", ha="right", color=col,
                   fontweight="bold", family="monospace")

# Legend at bottom
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=7.5,
          framealpha=0.95, bbox_to_anchor=(0.5, 0.0), columnspacing=1.5)

plt.savefig(FIG_DIR / "fig2_trend_timeseries.pdf", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "fig2_trend_timeseries.png", dpi=300, bbox_inches="tight")
plt.show()


## Figure 3: Cross-Site Comparison
4 separate panels, each with its own y-axis scale. Filled = significant, hatched = not significant.

In [ ]:
# ══════════════ FIGURE 3: CROSS-SITE COMPARISON ══════════════
trends_df = data.get("trends")
metrics = [
    ("center_lat", "Position\n(\u00b0/dec)", True),
    ("width_km",   "Width\n(km/dec)", False),
    ("speed_m_s",  "Speed\n" + r"(m s$^{-1}$/dec)", False),
    ("eke_m2_s2",  "EKE\n" + r"($\times 10^{-4}$ m$^{2}$ s$^{-2}$/dec)", False),
]
skeys = list(SITES.keys())
x = np.arange(len(skeys))
bw = 0.6
fig, axes = plt.subplots(1, 4, figsize=(13, 4))

for j, (metric, ylabel, add_km) in enumerate(metrics):
    ax = axes[j]
    vals = []
    for k, sk in enumerate(skeys):
        row = trends_df[(trends_df["site"] == sk) & (trends_df["metric"] == metric)]
        if len(row) == 0:
            continue
        s = row.iloc[0]["slope_per_decade"]
        sig_flag = row.iloc[0]["significant"]
        p_val = row.iloc[0]["p_value"]
        display_s = s * 1e4 if metric == "eke_m2_s2" else s
        vals.append(display_s)
        col = SITES[sk]["color"]
        if sig_flag:
            ax.bar(x[k], display_s, bw, color=col, ec="black", lw=0.6)
        else:
            ax.bar(x[k], display_s, bw, fc="white", ec=col, lw=1.5, hatch="///")

    # Set ylim with space for annotations
    if vals:
        ymin, ymax = min(min(vals), 0), max(max(vals), 0)
        rng = max(ymax - ymin, 0.01)
        ax.set_ylim(ymin - rng * 0.4, ymax + rng * 0.4)

    # Add annotations
    for k, sk in enumerate(skeys):
        row = trends_df[(trends_df["site"] == sk) & (trends_df["metric"] == metric)]
        if len(row) == 0:
            continue
        s = row.iloc[0]["slope_per_decade"]
        sig_flag = row.iloc[0]["significant"]
        p_val = row.iloc[0]["p_value"]
        display_s = s * 1e4 if metric == "eke_m2_s2" else s
        col = SITES[sk]["color"]
        star = sig_label(p_val)
        rng = ax.get_ylim()[1] - ax.get_ylim()[0]
        offset = rng * 0.04

        if add_km:
            km_val = s * DEG_TO_KM
            label_txt = f"{s:+.2f}\u00b0\n({km_val:+.1f} km)"
            if star:
                label_txt += f" {star}"
            yp = display_s + offset if display_s >= 0 else display_s - offset
            ax.text(x[k], yp, label_txt, ha="center",
                   va="bottom" if display_s >= 0 else "top",
                   fontsize=5.5, color=col, fontweight="bold")
        else:
            sign = "+" if display_s > 0 else ""
            val_txt = f"{sign}{display_s:.4f}" if metric == "speed_m_s" else (
                      f"{sign}{display_s:.1f}" if metric == "eke_m2_s2" else
                      f"{sign}{display_s:.2f}")
            if star:
                val_txt += f" {star}"
            yp = display_s + offset if display_s >= 0 else display_s - offset
            ax.text(x[k], yp, val_txt, ha="center",
                   va="bottom" if display_s >= 0 else "top",
                   fontsize=5.5, color=col, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels([SITES[sk]["name"] for sk in skeys], fontsize=6, rotation=35, ha="right")
    ax.axhline(0, color="black", lw=0.6)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(f"({chr(97+j)})", fontsize=10, fontweight="bold", loc="left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.tight_layout(w_pad=1.0)
plt.savefig(FIG_DIR / "fig3_crosssite_comparison.pdf", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "fig3_crosssite_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


## Figure 4: Forcing Context
(a) ERA5 10-m zonal wind time series + trends. (b) Argo temperature change between decades.

In [ ]:
# ══════════════ FIGURE 4: FORCING CONTEXT ══════════════
all_years = list(range(1993, 2026))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Panel (a) ──
ax = axes[0]
wt = data.get("wind_trends")
for sk, site in SITES.items():
    wk = f"wind_{sk}"
    if wk not in data or "u10" not in data[wk].columns:
        continue
    u_ann = data[wk]["u10"].resample("YS").mean()
    ax.plot(u_ann.index, u_ann.values, color=site["color"], lw=1.2,
           label=site["name"], marker="o", ms=2.5, zorder=3)
    if wt is not None:
        row = wt[(wt["site"] == sk) & (wt["metric"] == "u10")]
        if len(row) > 0:
            slope = row.iloc[0]["slope_per_decade"]
            sig_flag = row.iloc[0]["significant"]
            xn = np.arange(len(u_ann))
            ax.plot(u_ann.index, u_ann.values[0] + (slope / 10) * xn,
                   color=site["color"], lw=0.8,
                   ls="-" if sig_flag else "--", alpha=0.45, zorder=2)

# Annotate — bottom-left to avoid overlap with high SWIR line
if wt is not None:
    u10r = wt[wt["metric"] == "u10"].sort_values("site")
    for idx, (_, row) in enumerate(u10r.iterrows()):
        sk = row["site"]
        slope = row["slope_per_decade"]
        p = row["p_value"]
        star = sig_label(p)
        ann = f"{SITES[sk]['name']}: +{slope:.3f}" + r" m s$^{-1}$/dec"
        if star:
            ann += f" {star}"
        ax.text(0.02, 0.22 - idx * 0.06, ann, transform=ax.transAxes,
               fontsize=6.5, color=SITES[sk]["color"], fontweight="bold",
               bbox=dict(fc="white", ec="none", alpha=0.85))

ax.set_ylabel(r"Zonal wind speed (m s$^{-1}$)", fontsize=9)
ax.set_xlabel("Year", fontsize=9)
ax.set_title("(a)", fontsize=10, fontweight="bold", loc="left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticks([pd.Timestamp(f"{y}-01-01") for y in all_years])
ax.set_xticklabels([str(y) for y in all_years], rotation=45, ha="right", fontsize=5.5)
ax.legend(fontsize=6.5, loc="upper center", bbox_to_anchor=(0.5, -0.22),
         ncol=2, framealpha=0.95, columnspacing=1.0)

# ── Panel (b) ──
ax = axes[1]
argo = data.get("argo")
if argo is not None and len(argo) > 0:
    sks = argo["site"].values
    warming = argo["mean_warming_C"].values
    grad = argo["mean_abs_grad_change_C_100km"].values
    cols = [SITES[k]["color"] for k in sks]
    xp = np.arange(len(sks))
    bw = 0.35

    ax.bar(xp - bw/2, warming, bw, color=cols, ec="black", lw=0.5, alpha=0.85)
    ax2 = ax.twinx()
    ax2.bar(xp + bw/2, grad, bw, color=cols, ec="black", lw=0.5, alpha=0.4, hatch="...")

    for k in range(len(sks)):
        ax.text(xp[k] - bw/2, warming[k] + 0.003, f"{warming[k]:.3f}",
               ha="center", va="bottom", fontsize=6.5, color=cols[k], fontweight="bold")
        ax2.text(xp[k] + bw/2, grad[k] + 0.001, f"{grad[k]:.3f}",
                ha="center", va="bottom", fontsize=6.5, color=cols[k], fontweight="bold")

    ax.set_xticks(xp)
    ax.set_xticklabels([SITES[k]["name"] for k in sks], fontsize=7, rotation=25, ha="right")
    ax.set_ylabel("Warming at 500 dbar (\u00b0C)", fontsize=9)
    ax2.set_ylabel(r"$|\partial \overline{T}/\partial y|$ change ($^\circ$C / 100 km)", fontsize=9)
    ax.set_title("(b)", fontsize=10, fontweight="bold", loc="left")
    l1 = mpatches.Patch(fc="gray", ec="black", alpha=0.85, label="Warming (\u00b0C)")
    l2 = mpatches.Patch(fc="gray", ec="black", alpha=0.4, hatch="...",
                       label=r"$|\partial \overline{T}/\partial y|$ change")
    ax.legend(handles=[l1, l2], fontsize=7, loc="upper left")
    ax.spines["top"].set_visible(False)
    ax2.spines["top"].set_visible(False)

fig.tight_layout(w_pad=3.0)
fig.subplots_adjust(bottom=0.20)
plt.savefig(FIG_DIR / "fig4_forcing_context.pdf", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "fig4_forcing_context.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ══════════════ CLEANUP ══════════════
for key in list(data.keys()):
    if key.startswith("nc_") and hasattr(data[key], "close"):
        data[key].close()
print(f"All figures saved to: {FIG_DIR}")
